# 🔱 VoiceBatch Studio v2.8.0 - GitHub Ready
### Features: Unlimited Script, Anti-Sleep, Drive Integration, No Japanese Mixing.

In [ ]:
# @title 💤 Step 1: Anti-Sleep & Core Setup
import os
from IPython.display import display, Javascript

# Anti-Disconnect Script
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ Installing libraries for Python 3.12 compatibility...")
!pip install -q coqui-tts==0.22.0 --no-dependencies
!pip install -q trainer==0.0.36 coqpit==0.0.17 soundfile librosa pandas scipy encodec pydantic==2.8.2 gradio

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

os.makedirs("outputs", exist_ok=True)
print("✅ Setup Complete. Drive Connected.")

In [ ]:
# @title 🚀 Step 2: Launch App from Drive Models
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/" #

if os.path.exists(model_path + "model.pth"):
    print("⏳ Loading local model from Drive...")
    tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)
else:
    print("⚠️ Drive models not found. Ensure Step 1 transfer was successful.")

def voice_engine(text, audio_sample):
    # Unlimited Script Logic: Split by Hindi punctuation
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined = []
    for p in parts:
        if len(p.strip()) < 2: continue
        tts.tts_to_file(text=p, speaker_wav=audio_sample, language='hi', file_path='temp.wav')
        y, _ = librosa.load('temp.wav', sr=24000)
        combined.extend(y)
    
    sf.write('outputs/final.wav', np.array(combined), 24000)
    return 'outputs/final.wav'

gr.Interface(fn=voice_engine, inputs=[gr.Textbox(lines=10), gr.Audio(type='filepath')], outputs=gr.Audio()).launch(share=True)